In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler

from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

# ==============================
# LOAD DATA
# ==============================

dataset = pd.read_csv("../dataset/riemann_features.csv")

features = [
    "z_co_gram_lag_2",
    "z_gram",
    "z_gram_lag_1",
    "d_lag_13",
    "z_co_gram_lag_3",
    "z_co_gram_lag_1",
    "d_lag_14",
    "d_lag_1",
    "z_gram_lag_2",
    "d_lag_17"
]

X = dataset[features].values
y = dataset["distance"].values

# usar apenas 2000 pontos
X = X[:2000]
y = y[:2000]

split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# ==============================
# SCALE
# ==============================

scaler = MinMaxScaler(feature_range=(-1,1))

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==============================
# PCA (crucial para QML)
# ==============================

pca = PCA(n_components=6)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

N_QUBITS = X_train.shape[1]

# ==============================
# QUANTUM KERNEL
# ==============================

def create_quantum_kernel(n_qubits, reps=3, entanglement="circular", paulis=["ZZ","Z"]):
    feature_map = pauli_feature_map(
        feature_dimension=n_qubits,
        reps=reps,
        entanglement=entanglement,
        paulis=paulis
    )
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(
        sampler=sampler
    )
    quantum_kernel = FidelityQuantumKernel(
        feature_map=feature_map,
        fidelity=fidelity
    )
    return quantum_kernel

quantum_kernel = create_quantum_kernel(
    n_qubits=N_QUBITS,
    paulis=["ZZ","ZX","Z"],
    reps=3
)

# ==============================
# SVR COM KERNEL QUÂNTICO
# ==============================

svr = SVR(
    kernel=quantum_kernel.evaluate,
    C=15,
    epsilon=0.0008
)

print("Treinando QSVR...")

svr.fit(X_train, y_train)

pred = svr.predict(X_test)

# ==============================
# MÉTRICAS
# ==============================

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\n===== RESULTADOS QSVR =====")
print(f"RMSE: {rmse:.5f}")
print(f"R2: {r2:.5f}")

Treinando QSVR...


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler

from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer

# ==========================
# LOAD DATA
# ==========================

dataset = pd.read_csv("../dataset/riemann_features.csv")

features = [
    "z_co_gram_lag_2",
    "z_gram",
    "z_gram_lag_1",
    "d_lag_13",
    "z_co_gram_lag_3",
    "z_co_gram_lag_1",
    "d_lag_14",
    "d_lag_1",
    "z_gram_lag_2",
    "d_lag_17"
]

X = dataset[features].values
y = dataset["distance"].values

X = X[:2000]
y = y[:2000]

split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# ==========================
# SCALE
# ==========================

scaler = MinMaxScaler(feature_range=(-1,1))

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================
# PCA
# ==========================

pca = PCA(n_components=5)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

n_qubits = X_train.shape[1]

# ==========================
# FEATURE MAP
# ==========================

feature_map = pauli_feature_map(
    feature_dimension=n_qubits,
    reps=4,
    entanglement="circular",
    paulis=["Z","ZZ","ZX"]
)

sampler = StatevectorSampler()

fidelity = ComputeUncompute(
    sampler=sampler
)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    fidelity=fidelity
)

# ==========================
# TRAIN QUANTUM KERNEL
# ==========================

print("Training quantum kernel...")

trainer = QuantumKernelTrainer(
    quantum_kernel=quantum_kernel
)

kernel_result = trainer.fit(
    X_train,
    y_train
)

trained_kernel = kernel_result.quantum_kernel

# ==========================
# SVR
# ==========================

svr = SVR(
    kernel=trained_kernel.evaluate,
    C=20,
    epsilon=0.0005
)

print("Training QSVR...")

svr.fit(X_train, y_train)

pred = svr.predict(X_test)

# ==========================
# METRICS
# ==========================

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\n===== QSVR RESULTS =====")
print(f"RMSE: {rmse:.6f}")
print(f"R2: {r2:.6f}")